### the intention here is to build the workflow where we gather data response from the llm about whats in the pdf guia, that was proven before that it can be accurate, in notebook 04, and then compare with the field DS_PROCEDIMENTO and see if they match or not, generating another df with a aut_validacao_status. we can t keep ds contato , so in the workflow we check if theres a analysis form the analist or not for comparing only purposes of the past data

# Part 1

load the dataframe into memory and bring it to a valid input for unstructured

### DS_CONTATO, meaning the data from query finalizado, we dont care in production stage anymore, so we dont use this base anymore

In [1]:
import sys

# add app to path so we dont get the error ModuleNotFoundError: No module named 'app'
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from app.services.database.oracle import OracleService
from app.services.database.mariadb import MariaDBService
from app.utils.db_operations import load_query_from_file, execute_query_to_df
from app.utils.config import load_config
from app.utils.logger import get_logger


logger = get_logger(name=__name__)
config_vars = load_config()

In [2]:
# load the query strings
autorizacao_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_autorizacao.sql"
autorizacao_query_str = load_query_from_file(autorizacao_query_file)

procedimento_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_procedimento.sql"
procedimento_query_str = load_query_from_file(procedimento_query_file)

finalizado_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_pedido_finalizado_resposta.sql"
finalizado_query_str = load_query_from_file(finalizado_query_file)

aviso_cirurgia_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_aviso_cirurgia.sql"
aviso_cirurgia_query_str = load_query_from_file(aviso_cirurgia_query_file)

In [3]:
maria_db_procedimento_df = execute_query_to_df(db_service_class=MariaDBService(settings=config_vars), query=procedimento_query_str, fetch_limit=500)
oracle_db_autorizacao_df = execute_query_to_df(db_service_class=OracleService(settings=config_vars), query=autorizacao_query_str)
oracle_db_finalizado_df = execute_query_to_df(db_service_class=OracleService(settings=config_vars), query=finalizado_query_str)
maria_db_aviso_cirurgia_df = execute_query_to_df(db_service_class=MariaDBService(settings=config_vars), query=aviso_cirurgia_query_str)

{"timestamp": "2025-08-28T16:25:52", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ MariaDB connection established to\n                srvawsdb002.cow7tj30bxpl.us-east-1.rds.amazonaws.com:3306/", "filename": "mariadb.py", "lineno": 27}
{"timestamp": "2025-08-28T16:25:52", "level": "INFO", "name": "app.services.database.mariadb", "message": "Executing MariaDB query...", "filename": "mariadb.py", "lineno": 67}
{"timestamp": "2025-08-28T16:25:52", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ Fetched 280 rows from MariaDB.", "filename": "mariadb.py", "lineno": 75}
{"timestamp": "2025-08-28T16:25:52", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ MariaDB connection closed.", "filename": "mariadb.py", "lineno": 41}
{"timestamp": "2025-08-28T16:25:52", "level": "INFO", "name": "app.utils.db_operations", "message": "✅ Query executed successfully with 280 sample records", "filename": "db_operations.py", "lineno": 60}


## getting a single table with the info we need

In [4]:
import pandas as pd

maria_db_procedimento_df_processed = maria_db_procedimento_df.rename(columns={"surgical_order_id": "CD_AVISO_CIRURGIA", "procedure": "DS_PROCEDIMENTO"})
maria_db_procedimento_df_processed.drop(columns=["hospitalization_type"], inplace=True)
maria_db_procedimento_df_processed = maria_db_procedimento_df_processed[maria_db_procedimento_df_processed["DS_PROCEDIMENTO"].notna()]
maria_db_procedimento_df_processed.reset_index(drop=True, inplace=True)

autorizacao_no_need_cols = [col for col in oracle_db_autorizacao_df.columns if col not in ["CD_AVISO_CIRURGIA", "CD_GUIA", "CD_SENHA", "DS_GUIA_PATH"]]
oracle_db_autorizacao_df_processed = oracle_db_autorizacao_df.drop(columns=autorizacao_no_need_cols)
oracle_db_autorizacao_df_processed = oracle_db_autorizacao_df_processed[oracle_db_autorizacao_df_processed["DS_GUIA_PATH"].notna()]
oracle_db_autorizacao_df_processed.reset_index(drop=True, inplace=True)

finalizado_no_need_cols = [col for col in oracle_db_finalizado_df.columns if col not in ["CD_REGISTRO_VINCULADO", "DS_CONTATO"]]
oracle_db_finalizado_df_processed = oracle_db_finalizado_df.drop(columns=finalizado_no_need_cols)
oracle_db_finalizado_df_processed.rename(columns={"CD_REGISTRO_VINCULADO": "CD_AVISO_CIRURGIA"}, inplace=True)
oracle_db_finalizado_df_processed = oracle_db_finalizado_df_processed[oracle_db_finalizado_df_processed["DS_CONTATO"].notna()]
oracle_db_finalizado_df_processed.reset_index(drop=True, inplace=True)

maria_db_aviso_cirurgia_df.rename(columns={"surgical_order_id": "CD_AVISO_CIRURGIA"}, inplace=True)
keep_cols = ["CD_AVISO_CIRURGIA", "health_insurance_name"]
maria_db_aviso_cirurgia_df = maria_db_aviso_cirurgia_df[keep_cols]
maria_db_aviso_cirurgia_df.reset_index(drop=True, inplace=True)

In [5]:
merged_df_autorizacao = pd.merge(
    maria_db_procedimento_df_processed,
    oracle_db_autorizacao_df_processed,
    on="CD_AVISO_CIRURGIA",
    how="inner"
)

merged_df_autorizacao_final = pd.merge(
    merged_df_autorizacao,
    oracle_db_finalizado_df_processed,
    on="CD_AVISO_CIRURGIA",
    how="inner"
)

final_extracted_df = pd.merge(
    merged_df_autorizacao_final,
    maria_db_aviso_cirurgia_df,
    on="CD_AVISO_CIRURGIA",
    how="inner"
)

In [6]:
final_extracted_df.head(3)

,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name
0,857469,"[{""code"":30907136,""description"":""VARIZES - TRA...",19495507.0,J5VEYT7,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO
1,857883,"[{""code"":30205050,""description"":""AMIGDALECTOMI...",19506292.0,J5VEW29,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO
2,858036,"[{""code"":30205247,""description"":""UVULOPALATO-F...",19508906.0,J5VEWF9,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Parcialmente autorizado\n...,BRADESCO


In [7]:
# entering the pdf link, downloading the pdf, adding as another col
import requests
import numpy as np
def fetch_pdf_bytes(url):
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200 and 'application/pdf' in response.headers.get('content-type', ''):
            return response.content
        else:
            return np.nan
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return np.nan

# Apply to all links in DS_GUIA_PATH
final_extracted_df['DS_PDF_BYTES'] = final_extracted_df['DS_GUIA_PATH'].apply(fetch_pdf_bytes)
final_extracted_df.head(3)

,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name,DS_PDF_BYTES
0,857469,"[{""code"":30907136,""description"":""VARIZES - TRA...",19495507.0,J5VEYT7,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...
1,857883,"[{""code"":30205050,""description"":""AMIGDALECTOMI...",19506292.0,J5VEW29,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...
2,858036,"[{""code"":30205247,""description"":""UVULOPALATO-F...",19508906.0,J5VEWF9,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Parcialmente autorizado\n...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...


In [8]:
# Create a new column with clickable links for DS_GUIA_PATH
def make_clickable(url):
    if pd.notna(url):
        return f'<a href="{url}" target="_blank">{url}</a>'
    return ""
final_extracted_df['DS_GUIA_PATH_CLICKABLE'] = final_extracted_df['DS_GUIA_PATH'].apply(make_clickable)
from IPython.display import display, HTML
display(HTML(final_extracted_df[['DS_GUIA_PATH', 'DS_GUIA_PATH_CLICKABLE']].head(10).to_html(escape=False)))

,DS_GUIA_PATH,DS_GUIA_PATH_CLICKABLE
0,https://cdns.overmind.ai/autorizacao-bradesco-121223221-1752237144964.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121223221-1752237144964.pdf
1,https://cdns.overmind.ai/autorizacao-bradesco-121223228-1753454043822.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121223228-1753454043822.pdf
2,https://cdns.overmind.ai/autorizacao-bradesco-121223237-1753796587472.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121223237-1753796587472.pdf
3,https://cdns.overmind.ai/autorizacao-sulamerica-203941226-1753068369928.pdf,https://cdns.overmind.ai/autorizacao-sulamerica-203941226-1753068369928.pdf
4,https://cdns.overmind.ai/autorizacao-bradesco-121576086-1753117675678.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121576086-1753117675678.pdf
5,https://cdns.overmind.ai/autorizacao-sulamerica-204656188-1754489468918.pdf,https://cdns.overmind.ai/autorizacao-sulamerica-204656188-1754489468918.pdf
6,https://cdns.overmind.ai/autorizacao-bradesco-121612505-1753983787863.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121612505-1753983787863.pdf
7,https://cdns.overmind.ai/autorizacao-sulamerica-204937675-1754852532865.pdf,https://cdns.overmind.ai/autorizacao-sulamerica-204937675-1754852532865.pdf
8,https://cdns.overmind.ai/autorizacao-sulamerica-205226366-1755799999288.pdf,https://cdns.overmind.ai/autorizacao-sulamerica-205226366-1755799999288.pdf
9,https://cdns.overmind.ai/autorizacao-bradesco-121847650-1754590032734.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121847650-1754590032734.pdf


# part 2: inputing data to technique, observing the outputed data

In [9]:
# trying to use the mudular code to get the workflow feeling
import sys

# add app to path so we dont get the error ModuleNotFoundError: No module named 'app'
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from app.utils.process_images import process_blobs
from app.utils.textract_service import TextractPDFAnalyzer
from app.utils.llm_service import AnthropicLLMService
from app.utils.aws_services_handler import create_boto3_client
from app.utils.config import load_config, AppConstants
from app.utils.logger import get_logger
from app.utils.system_prompts.autorizacao_prompt import Prompts


logger = get_logger(name=__name__)
config_vars = load_config()

In [10]:
app_constants = AppConstants()
textract_client = create_boto3_client("textract", config_vars)
bedrock_client = create_boto3_client("bedrock-runtime", config_vars)
s3_client = create_boto3_client("s3", config_vars)
bucket_name = "autorizacoes"

# Initialize the TextractPDFAnalyzer
pdf_analyzer = TextractPDFAnalyzer(
    textract_client=textract_client,
    s3_client=s3_client,
    bucket_name="autorizacoes-textract",
    bucket_folder="guias-pdf"
)

autorizacao_prompt = Prompts.autorizacao_extraction_prompt


llm_instance = AnthropicLLMService(
    model_id=app_constants.BEDROCK_DEFAULT_MODEL_ID,
    model_version=app_constants.BEDROCK_DEFAULT_MODEL_VERSION,
    client=bedrock_client,
    system_prompt=autorizacao_prompt,
    max_tokens=app_constants.MAX_TOKENS,
    temperature=app_constants.TEMPERATURE,
    budget_tokens=app_constants.BUDGET_TOKENS
)

{"timestamp": "2025-08-28T16:29:51", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente TEXTRACT para a região: us-east-1...", "filename": "aws_services_handler.py", "lineno": 17}
{"timestamp": "2025-08-28T16:29:51", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Cliente TEXTRACT criado com sucesso.", "filename": "aws_services_handler.py", "lineno": 26}
{"timestamp": "2025-08-28T16:29:51", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "aws_services_handler.py", "lineno": 17}
{"timestamp": "2025-08-28T16:29:51", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Cliente BEDROCK-RUNTIME criado com sucesso.", "filename": "aws_services_handler.py", "lineno": 26}
{"timestamp": "2025-08-28T16:29:51", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente S3 para a região: us-east-1...

In [42]:
# llm processing

from typing import Dict, Tuple
# Process rows from the dataframe 
def llm_extraction(df: pd.DataFrame, limit=None) -> Tuple[Dict, Dict]:
    """
    Extract data from PDFs using Textract and process with LLM.
    
    Args:
        df (pd.DataFrame): DataFrame containing PDF information
        limit (int, optional): Limit number of rows to process
        
    Returns:
        Tuple[Dict, Dict]: LLM results and Textract results dictionaries
    """
    if limit:
        df = df[:limit]
    textract_results = {}
    llm_results = {}
    for index, row in df.iterrows():
        pdf_bytes = row['DS_PDF_BYTES']
        file_id = str(row['CD_AVISO_CIRURGIA'])
        
        if pd.isna(pdf_bytes):
            print(f"Skipping row {index}: No PDF bytes available")
            continue
        
        # Extract forms data using Textract
        forms_data = pdf_analyzer.extract_forms_data_from_pdf(
            pdf_bytes=pdf_bytes,
            file_id=file_id
        )
        textract_results[file_id] = forms_data
        
        
        # Process with LLM
        forms_data_str = str(forms_data)
        llm_response = llm_instance.invoke_model(input_str=forms_data_str)
        llm_results[file_id] = llm_response
    return llm_results, textract_results

def llm_dict_to_final_df(llm_results: Dict, textract_results: Dict, df: pd.DataFrame, limit=None) -> pd.DataFrame:
    """
    Convert LLM and Textract results dictionaries to final merged DataFrame.
    
    Args:
        llm_results (Dict): Dictionary with LLM processing results
        textract_results (Dict): Dictionary with Textract extraction results
        df (pd.DataFrame): Original DataFrame to merge with
        limit (int, optional): Limit number of rows from original DataFrame
        
    Returns:
        pd.DataFrame: Merged DataFrame with LLM and Textract results
    """
    if limit:
        df = df[:limit]

    llm_results_df = pd.DataFrame.from_dict(llm_results, orient="index")
    llm_results_df.index.name = "CD_AVISO_CIRURGIA"
    llm_results_df = llm_results_df.reset_index()

    # Expand LLM results into separate columns
    llm_expanded_df = pd.json_normalize(llm_results_df.iloc[:, 1:].to_dict('records'))
    llm_expanded_df['CD_AVISO_CIRURGIA'] = llm_results_df['CD_AVISO_CIRURGIA']

    # Add suffix to distinguish LLM columns
    llm_expanded_df = llm_expanded_df.add_suffix('_llm').rename(columns={'CD_AVISO_CIRURGIA_llm': 'CD_AVISO_CIRURGIA'})

    # Convert CD_AVISO_CIRURGIA to int to match the original dataframe type
    llm_expanded_df['CD_AVISO_CIRURGIA'] = llm_expanded_df['CD_AVISO_CIRURGIA'].astype(int)
    
    # Map textract results based on CD_AVISO_CIRURGIA values
    # Convert the keys in textract_results to int for proper mapping
    textract_results_int_keys = {int(k): v for k, v in textract_results.items()}
    llm_expanded_df['textract_results'] = llm_expanded_df['CD_AVISO_CIRURGIA'].map(textract_results_int_keys)

    # Ensure original df has the correct data type
    df = df.copy()
    df['CD_AVISO_CIRURGIA'] = df['CD_AVISO_CIRURGIA'].astype(int)

    # Merge df with llm results
    merged_df = df.merge(
        llm_expanded_df,
        on='CD_AVISO_CIRURGIA',
        how='inner'
    )
    # Inner join with original dataframe
    return merged_df

In [ ]:
llm_results, textract_results = llm_extraction(df=final_extracted_df, limit=5)
llm_results_df_run = llm_dict_to_final_df(llm_results=llm_results, textract_results=textract_results, df=final_extracted_df, limit=10)

print(f"Processing completed! Created textract_eval_df with {len(llm_results_df_run)} rows")
print(f"LLM columns added: {[col for col in llm_results_df_run.columns if col.endswith('_llm')]}")
llm_results_df_run.head()



{"timestamp": "2025-08-28T17:01:53", "level": "INFO", "name": "app.utils.textract_service", "message": "PDF uploaded to S3: s3://autorizacoes-textract/guias-pdf/textract_pdfs/857469_20250828_170154_9b415629.pdf", "filename": "textract_service.py", "lineno": 177}
{"timestamp": "2025-08-28T17:01:54", "level": "INFO", "name": "app.utils.textract_service", "message": "Started PDF analysis with JobId: e8a3bd99224601d6c68af1fef81a419af7c5ef9703757fe45ffbd5ab9cfced5f", "filename": "textract_service.py", "lineno": 210}
{"timestamp": "2025-08-28T17:01:54", "level": "INFO", "name": "app.utils.textract_service", "message": "PDF analysis in progress for JobId: e8a3bd99224601d6c68af1fef81a419af7c5ef9703757fe45ffbd5ab9cfced5f", "filename": "textract_service.py", "lineno": 264}
{"timestamp": "2025-08-28T17:01:59", "level": "INFO", "name": "app.utils.textract_service", "message": "PDF analysis in progress for JobId: e8a3bd99224601d6c68af1fef81a419af7c5ef9703757fe45ffbd5ab9cfced5f", "filename": "textra

Processing completed! Created textract_eval_df with 5 rows
LLM columns added: ['procedimentos_autorizados_llm', 'paciente_llm', 'codigo_autorizado_llm', 'senha_llm', 'validade_senha_llm', 'data_solicitacao_llm', 'observacoes_opme_llm', 'observacoes_gerais_llm', 'profissional_solicitante_llm']


In [58]:
llm_results_df = llm_results_df_run.copy()
llm_results_df.loc[3, 'CD_AVISO_CIRURGIA']

np.int64(859112)

In [59]:
# value of the col DS_PROCEDIMENTO of the 4 row:
llm_results_df.loc[3, 'DS_PROCEDIMENTO']

'[{"code":30501350,"description":"RINOSSEPTOPLASTIA","isMain":true,"pro_fat_id":"30501350","procedure_code":30501350,"quantity":1,"surgery_id":1187},{"code":30501458,"description":"TURBINECTOMIA OU TURBINOPLASTIA - UNILATERAL","isMain":false,"pro_fat_id":"30501458","procedure_code":30501458,"quantity":2,"surgery_id":1191}]'

In [60]:
# value of the col ds_contato of the 4 row:
llm_results_df.loc[3, 'DS_CONTATO']

'\n      Procedimento Autorizado\n      Paciente: LEANDRO SALOMAO ROSA\n      CÓDIGO AUTORIZADO:\n      01009001 x 1\n,30501350 x 1\n,30501458 x 2\n\n      \n      SENHA: 5045663575\n      VALIDADE DA SENHA: 19/08/2025\n      OBSERVAÇÕES OPME: Pedido sem OPME\n      OBSERVAÇÕES GERAIS: Diária: 01\n      NOME USUÁRIO FINALIZOU O PEDIDO: TELMA\n    '

In [61]:
llm_results_df.loc[3, 'codigo_autorizado_llm']

'65300114 x 01, 30501458 x 02, 30501350 x 01'

In [62]:
# Create a new column with clickable links for DS_GUIA_PATH
def make_clickable(url):
    if pd.notna(url):
        return f'<a href="{url}" target="_blank">{url}</a>'
    return ""
llm_results_df['DS_GUIA_PATH_CLICKABLE'] = llm_results_df['DS_GUIA_PATH'].apply(make_clickable)
from IPython.display import display, HTML
display(HTML(llm_results_df[['CD_AVISO_CIRURGIA','DS_GUIA_PATH', 'DS_GUIA_PATH_CLICKABLE']].head(10).to_html(escape=False)))

,CD_AVISO_CIRURGIA,DS_GUIA_PATH,DS_GUIA_PATH_CLICKABLE
0,857469,https://cdns.overmind.ai/autorizacao-bradesco-121223221-1752237144964.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121223221-1752237144964.pdf
1,857883,https://cdns.overmind.ai/autorizacao-bradesco-121223228-1753454043822.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121223228-1753454043822.pdf
2,858036,https://cdns.overmind.ai/autorizacao-bradesco-121223237-1753796587472.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121223237-1753796587472.pdf
3,859112,https://cdns.overmind.ai/autorizacao-sulamerica-203941226-1753068369928.pdf,https://cdns.overmind.ai/autorizacao-sulamerica-203941226-1753068369928.pdf
4,861179,https://cdns.overmind.ai/autorizacao-bradesco-121576086-1753117675678.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121576086-1753117675678.pdf


# Part 3: comparing the data from the llm with the field DS_PROCEDIMENTO, generating a new field validacao_status with the values "match" or "not match"

The new fields cols would be  validacao_status_codigo_procedimento, validacao_status_descricao_procedimento


In [72]:
# Part 3: comparing the data from the llm with the field DS_PROCEDIMENTO, generating a new field validacao_status with the values "match" or "not match"

# The new fields cols would be  validacao_status_codigo_procedimento, validacao_status_descricao_procedimento
# example of the field DS_PROCEDIMENTO:
# '[{"code":30205050,"description":"AMIGDALECTOMIA DAS PLATINAS","isMain":true,"pro_fat_id":"30205050","procedure_code":30205050,"quantity":1,"surgery_id":1227},{"code":30205271,"description":"ADENOIDECTOMIA POR VIDEOENDOSCOPIA","isMain":false,"pro_fat_id":"30205271","procedure_code":30205271,"quantity":1,"surgery_id":5974},{"code":30205069,"description":"AMIDALECTOMIA LINGUAL","isMain":false,"pro_fat_id":"30205069","procedure_code":30205069,"quantity":1,"surgery_id":1236},{"code":30501458,"description":"TURBINECTOMIA OU TURBINOPLASTIA - UNILATERAL","isMain":false,"pro_fat_id":"30501458","procedure_code":30501458,"quantity":2,"surgery_id":1191}]'
# example of the field 'codigo_autorizado_llm':
# '30205050 x 1, 30205069 x 1, 30205271 x 1, 30501458 x 2'




import pandas as pd
import json

import pandas as pd
import json
from typing import List, Tuple

def validate_codigo_procedimento(
    ds_procedimento_col: List[str], 
    codigo_autorizado_llm_col: List[str]
) -> Tuple[List[str], List[str]]:
    """
    Compares DS_PROCEDIMENTO and codigo_autorizado_llm columns to generate validation status and explanations.

    Args:
        ds_procedimento_col (List[str]): List of DS_PROCEDIMENTO JSON strings.
        codigo_autorizado_llm_col (List[str]): List of LLM output strings (e.g., '30205050 x 1, 30205069 x 1').

    Returns:
        Tuple[List[str], List[str]]: Tuple containing:
            - List of validation statuses ("match" or "not match") for each row
            - List of explanations describing what doesn't match
    """
    def parse_ds_procedimento(json_str):
        try:
            data_list = json.loads(json_str)
            return set(
                (str(item.get('procedure_code', item.get('code'))), str(item['quantity']))
                for item in data_list
            )
        except (json.JSONDecodeError, TypeError):
            return set()

    def parse_llm_data(llm_str):
        if pd.isna(llm_str):
            return set()
        try:
            items = llm_str.split(', ')
            return set(
                (item.split(' x ')[0], item.split(' x ')[1])
                for item in items
            )
        except IndexError:
            return set()

    def compare_data_with_explanation(ds_set, llm_set):
        if ds_set.issubset(llm_set):
            return "correspondente", "Todos os procedimentos de DS_PROCEDIMENTO estão presentes na saída do LLM"
        
        # Find missing procedures
        missing_in_llm = ds_set - llm_set
        extra_in_llm = llm_set - ds_set
        
        explanations = []
        
        if missing_in_llm:
            missing_codes = [f"código {code} x {qty}" for code, qty in missing_in_llm]
            explanations.append(f"Ausente no LLM: {', '.join(missing_codes)}")
        
        if extra_in_llm:
            extra_codes = [f"código {code} x {qty}" for code, qty in extra_in_llm]
            explanations.append(f"Extra no LLM: {', '.join(extra_codes)}")
        
        return "não correspondente", "; ".join(explanations)

    results = [
        compare_data_with_explanation(
            parse_ds_procedimento(ds_proc), 
            parse_llm_data(llm_val)
        )
        for ds_proc, llm_val in zip(ds_procedimento_col, codigo_autorizado_llm_col)
    ]
    
    statuses = [result[0] for result in results]
    explanations = [result[1] for result in results]
    
    return statuses, explanations



In [73]:
# Get both validation status and explanation
validation_status, validation_explanation = validate_codigo_procedimento(
    llm_results_df['DS_PROCEDIMENTO'].tolist(),
    llm_results_df['codigo_autorizado_llm'].tolist()
)

# Add both columns to the dataframe
llm_results_df['validacao_status_codigo_procedimento'] = validation_status
llm_results_df['validacao_explanation_codigo_procedimento'] = validation_explanation

llm_results_df.head()

,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name,DS_PDF_BYTES,DS_GUIA_PATH_CLICKABLE,procedimentos_autorizados_llm,...,codigo_autorizado_llm,senha_llm,validade_senha_llm,data_solicitacao_llm,observacoes_opme_llm,observacoes_gerais_llm,profissional_solicitante_llm,textract_results,validacao_status_codigo_procedimento,validacao_explanation_codigo_procedimento
0,857469,"[{""code"":30907136,""description"":""VARIZES - TRA...",19495507.0,J5VEYT7,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...",[VARIZES TRATAMENTO CIRURGICO DE DOIS MEMBROS],...,30907136 x 1,J5VEYT7,2026-01-07,2025-07-11,Pedido sem OPME,None,MATEUS ALVES BORGES CRISTINO,"{'forms': {'16 Número do Conselho': '43236', '...",correspondente,Todos os procedimentos de DS_PROCEDIMENTO estã...
1,857883,"[{""code"":30205050,""description"":""AMIGDALECTOMI...",19506292.0,J5VEW29,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...","[AMIGDALECTOMIA DAS PALATINAS, AMIGDALECTOMIA ...",...,"30205050 x 1, 30205069 x 1, 30205271 x 1, 3050...",J5VEW29,2026-01-07,2025-07-11,Pedido sem OPME,Solicitacao de autorizacao,AURELIA ALBUQUERQUE MARTINS,"{'forms': {'16 Número do Conselho': '41848', '...",correspondente,Todos os procedimentos de DS_PROCEDIMENTO estã...
2,858036,"[{""code"":30205247,""description"":""UVULOPALATO-F...",19508906.0,J5VEWF9,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Parcialmente autorizado\n...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...","[AMIGDALECTOMIA DAS PALATINAS, AMIGDALECTOMIA ...",...,"30205050 x 1, 30205069 x 1, 30205247 x 1, 3050...",J5VEWF9,2026-01-07,2025-07-11,Pedido sem OPME,Solicitacao de autorizacao,LUCAS EDUARDO DE OLIVEIRA,"{'forms': {'16 Número do Conselho': '81474', '...",não correspondente,"Ausente no LLM: código 30502357 x 2, código 30..."
3,859112,"[{""code"":30501350,""description"":""RINOSSEPTOPLA...",19537543.0,5045663575,https://cdns.overmind.ai/autorizacao-sulameric...,\n Procedimento Autorizado\n Pacient...,SUL AMERICA,b'%PDF-1.4\n%\xf6\xe4\xfc\xdf\n1 0 obj\n<<\n/T...,"<a href=""https://cdns.overmind.ai/autorizacao-...","[SEPTOPLASTIA TURBINECTOMIA, TURBINECTOMIA OU ...",...,"65300114 x 01, 30501458 x 02, 30501350 x 01",5045663575,2025-08-19,2025-07-14,Pedido sem OPME,None,None,{'forms': {'Produto': '545 EMPRESARIAL AMB HOS...,não correspondente,"Ausente no LLM: código 30501458 x 2, código 30..."
4,861179,"[{""code"":31009166,""description"":""HERNIORRAFIA ...",19595729.0,JNMLM61,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...",[HERNIORRAFIA UMBILICAL],...,31009166 x 1,JNMLM61,2026-01-17,2025-07-21,Pedido sem OPME,None,PEDRO HENRIQUE VILELA MOREIRA,"{'forms': {'16 Número do Conselho': '58880', '...",correspondente,Todos os procedimentos de DS_PROCEDIMENTO estã...


In [74]:
llm_results_df.loc[2, 'DS_PROCEDIMENTO']

'[{"code":30205247,"description":"UVULOPALATO-FARINGOPLASTIA","isMain":true,"pro_fat_id":"30205247","procedure_code":30205247,"quantity":1,"surgery_id":1239},{"code":30205050,"description":"AMIGDALECTOMIA DAS PLATINAS","isMain":false,"pro_fat_id":"30205050","procedure_code":30205050,"quantity":1,"surgery_id":1227},{"code":30205069,"description":"AMIDALECTOMIA LINGUAL","isMain":false,"pro_fat_id":"30205069","procedure_code":30205069,"quantity":1,"surgery_id":1236},{"code":30501369,"description":"SEPTOPLASTIA (QUALQUER TECNICA SEM VIDEO)","isMain":false,"pro_fat_id":"30501369","procedure_code":30501369,"quantity":1,"surgery_id":2006},{"code":30502349,"description":"SINUSOTOMIA ESFENOIDAL POR VIDEOENDOSCOP","isMain":false,"pro_fat_id":"30502349","procedure_code":30502349,"quantity":2,"surgery_id":5931},{"code":30502357,"description":"SINUSOTOMIA FRONTAL INTRANASAL POR VIDEOENDOSCOPIA","isMain":false,"pro_fat_id":"30502357","procedure_code":30502357,"quantity":2,"surgery_id":5969},{"code":

In [75]:
llm_results_df.loc[2, 'codigo_autorizado_llm']

'30205050 x 1, 30205069 x 1, 30205247 x 1, 30501067 x 2, 30501369 x 1, 30501458 x 2, 30502292 x 0, 30502314 x 2, 30502349 x 0, 30502357 x 0'

In [76]:
llm_results_df.loc[2, 'validacao_explanation_codigo_procedimento']

'Ausente no LLM: código 30502357 x 2, código 30502292 x 2, código 30502349 x 2; Extra no LLM: código 30502292 x 0, código 30502349 x 0, código 30502357 x 0'

In [77]:
llm_results_df.loc[2, 'validade_senha_llm']

Timestamp('2026-01-07 00:00:00')

In [78]:
llm_results_df.loc[2, 'textract_results']

{'forms': {'16 Número do Conselho': '81474',
  '14 Nome do Profissional Solicitante': 'LUCAS EDUARDO DE OLIVEIRA',
  '20 Nome do Hospital/ Local Solicitado': 'HOSPITAL MATER DEI',
  '19 Código na Operadora / CNPJ': '450510',
  '17 UF': 'MG',
  '15 Conselho Profissional': 'CRM',
  '1 Registro ANS': '005711',
  '25 Qtde. Diárias': '1',
  '9 Atendimento a RN': 'Não',
  '3 Número da Guia Atribuído pela Operadora': '121223237',
  '23 -Tipo de': '2',
  '7 Número da Carteira': '973870074969002',
  '24 Regime de Internação': 'HOSPITALAR',
  '22 Caráter do': 'ELETIVO',
  'Senha': 'J5VEWF9',
  '21 Data Sugerida para Internação (Real)': '12/07/2025',
  '10 Nome': 'DIONE RAIMUNDO CARVALHO PINTO',
  '13 Nome do Contratado': 'HOSPITAL MATER DEI',
  '4 Data da Autorização': '29/07/2025',
  '12 Código na Operadora': '450510',
  '18 Código CBO': '292 MEDICO CLINICO',
  'Gerado em:': '29/07/2025 10:43',
  '34': 'Tabela',
  '29 CID 10 Principal': '30 CID 10 (2) 31 CID 10 (3) 32 CID 10 (4) 33 Indicação de

In [79]:
import pandas as pd
from datetime import timedelta
import numpy as np

# Replace string "None" with actual NaN values before processing
llm_results_df['validade_senha_llm'] = llm_results_df['validade_senha_llm'].replace('None', np.nan)

# Ensure date columns are in datetime format, coercing errors to NaT (Not a Time)
# This handles cases where the date might be None or in an invalid format
llm_results_df['data_solicitacao_llm'] = pd.to_datetime(llm_results_df['data_solicitacao_llm'], errors='coerce', dayfirst=True)
llm_results_df['validade_senha_llm'] = pd.to_datetime(llm_results_df['validade_senha_llm'], errors='coerce', dayfirst=True)

# Define the condition for the update
condition = (
    (llm_results_df['health_insurance_name'] == 'BRADESCO') &
    (llm_results_df['validade_senha_llm'].isna()) &
    (llm_results_df['data_solicitacao_llm'].notna())
)

# Apply the update where the condition is true
llm_results_df.loc[condition, 'validade_senha_llm'] = llm_results_df.loc[condition, 'data_solicitacao_llm'] + timedelta(days=180)

llm_results_df.head()


,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name,DS_PDF_BYTES,DS_GUIA_PATH_CLICKABLE,procedimentos_autorizados_llm,...,codigo_autorizado_llm,senha_llm,validade_senha_llm,data_solicitacao_llm,observacoes_opme_llm,observacoes_gerais_llm,profissional_solicitante_llm,textract_results,validacao_status_codigo_procedimento,validacao_explanation_codigo_procedimento
0,857469,"[{""code"":30907136,""description"":""VARIZES - TRA...",19495507.0,J5VEYT7,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...",[VARIZES TRATAMENTO CIRURGICO DE DOIS MEMBROS],...,30907136 x 1,J5VEYT7,2026-01-07,2025-07-11,Pedido sem OPME,None,MATEUS ALVES BORGES CRISTINO,"{'forms': {'16 Número do Conselho': '43236', '...",correspondente,Todos os procedimentos de DS_PROCEDIMENTO estã...
1,857883,"[{""code"":30205050,""description"":""AMIGDALECTOMI...",19506292.0,J5VEW29,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...","[AMIGDALECTOMIA DAS PALATINAS, AMIGDALECTOMIA ...",...,"30205050 x 1, 30205069 x 1, 30205271 x 1, 3050...",J5VEW29,2026-01-07,2025-07-11,Pedido sem OPME,Solicitacao de autorizacao,AURELIA ALBUQUERQUE MARTINS,"{'forms': {'16 Número do Conselho': '41848', '...",correspondente,Todos os procedimentos de DS_PROCEDIMENTO estã...
2,858036,"[{""code"":30205247,""description"":""UVULOPALATO-F...",19508906.0,J5VEWF9,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Parcialmente autorizado\n...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...","[AMIGDALECTOMIA DAS PALATINAS, AMIGDALECTOMIA ...",...,"30205050 x 1, 30205069 x 1, 30205247 x 1, 3050...",J5VEWF9,2026-01-07,2025-07-11,Pedido sem OPME,Solicitacao de autorizacao,LUCAS EDUARDO DE OLIVEIRA,"{'forms': {'16 Número do Conselho': '81474', '...",não correspondente,"Ausente no LLM: código 30502357 x 2, código 30..."
3,859112,"[{""code"":30501350,""description"":""RINOSSEPTOPLA...",19537543.0,5045663575,https://cdns.overmind.ai/autorizacao-sulameric...,\n Procedimento Autorizado\n Pacient...,SUL AMERICA,b'%PDF-1.4\n%\xf6\xe4\xfc\xdf\n1 0 obj\n<<\n/T...,"<a href=""https://cdns.overmind.ai/autorizacao-...","[SEPTOPLASTIA TURBINECTOMIA, TURBINECTOMIA OU ...",...,"65300114 x 01, 30501458 x 02, 30501350 x 01",5045663575,2025-08-19,2025-07-14,Pedido sem OPME,None,None,{'forms': {'Produto': '545 EMPRESARIAL AMB HOS...,não correspondente,"Ausente no LLM: código 30501458 x 2, código 30..."
4,861179,"[{""code"":31009166,""description"":""HERNIORRAFIA ...",19595729.0,JNMLM61,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...",[HERNIORRAFIA UMBILICAL],...,31009166 x 1,JNMLM61,2026-01-17,2025-07-21,Pedido sem OPME,None,PEDRO HENRIQUE VILELA MOREIRA,"{'forms': {'16 Número do Conselho': '58880', '...",correspondente,Todos os procedimentos de DS_PROCEDIMENTO estã...
